In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
import regex
from plotly.subplots import make_subplots
from jupyter_dash import JupyterDash
from dash import Dash, dcc, html, Input, Output, dash_table, callback
import dash_bootstrap_components as dbc

In [3]:
dane=pd.read_csv('dane_bkg.csv', index_col=0)
dane_rh=pd.read_csv('dane_rh.csv', index_col=0)
dane_compton=pd.read_csv('dane_compton.csv', index_col=0)
energia=[ round(x*0.02009, 2) for x in range(2048)]
pierwiastki=pd.read_csv('pierwiastki.csv', index_col=0)
pierwiastki_rh=pd.read_csv('pierwiastki_rh.csv', index_col=0)
pierwiastki_compton=pd.read_csv('pierwiastki_compton.csv', index_col=0)

In [4]:
dane

,1f.2,1af.2,2f.1,2f.1b,3f.1,3,4f.1,4f.1b,7f.1,7,...,325,326,327,328,329,330,331,332,334f.2,336f.8
1,506.313636,455.032197,504.675758,478.007576,469.748864,506.524242,509.674242,463.873485,508.749621,481.401136,...,505.551515,509.065152,484.445833,486.783333,492.712121,523.464773,526.312121,455.749242,458.895455,463.887879
2,417.735227,409.619697,425.256818,437.005682,437.486364,414.893182,463.255682,462.693182,442.362121,437.113636,...,379.731818,439.548864,423.312500,424.837500,428.284091,390.360227,402.734091,448.124242,456.421591,398.165909
3,167.156818,216.067424,206.597348,182.003788,212.384091,203.262121,196.837121,214.462121,153.978030,204.815909,...,181.487879,174.032576,159.208333,231.858333,154.856061,194.906818,133.156061,160.906061,195.437689,194.443939
4,10.578409,30.533712,0.000000,9.001894,27.692045,0.000000,0.000000,17.231061,50.489015,39.907955,...,21.243939,5.516288,20.104167,0.000000,0.000000,16.453409,2.578030,39.453030,0.000000,0.000000
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2044,9.376515,15.556250,2.560180,10.741477,10.804545,9.403030,6.875000,5.875000,15.750000,0.650568,...,5.000000,8.221212,7.217614,12.500000,0.000000,8.203220,0.250000,7.317803,7.020833,0.265909
2045,6.688258,1.667188,4.055303,3.883049,6.072727,3.836364,10.937500,0.937500,0.000000,7.676136,...,3.000000,9.831818,11.360985,12.290909,9.700000,3.032765,4.875000,8.635606,7.758333,2.109470
2046,0.000000,9.778125,5.527652,9.024621,5.340909,7.963258,0.000000,0.000000,9.614773,7.785795,...,9.000000,0.442424,9.551231,1.145455,8.400000,3.084564,1.500000,4.425758,3.508333,0.000000
2047,5.909801,8.889063,0.000000,3.179830,5.609091,8.481629,17.500000,1.529545,12.229545,1.895455,...,6.000000,10.053030,1.741477,0.000000,8.300000,4.136364,1.125000,1.017045,6.537500,5.031818


In [5]:
app = Dash(__name__, external_stylesheets=[dbc.themes.MINTY])#
#server = app.server

In [6]:
app.layout =dbc.Container(
    [
    html.H1('DRANGSONG Manuscripts -XRF analysis', style={'marginTop':'50px'}),
    html.H2('Wybierz próbki do wyświetlenia', style={'marginTop':'50px'}),
    html.Div([
        dcc.Checklist(
            options=dane.columns,
            value=['1f.2'],
            id='sample',
            style={'display': 'grid', 'grid-template-columns': 'repeat(20, 1fr)'}
        )], style={'width':'100%'}),
    html.H2('Wybierz dane:', style={'marginTop':'50px'}),
    dcc.RadioItems(
        options=[
            {'label': 'dane nieznormalizowane', 'value': 'dane'},
            {'label': 'dane znormalizowane do rodu', 'value': 'dane_rh'},
            {'label': 'dane znormalizowane do Comptona', 'value': 'dane_compton'}
        ],
        value='dane_rh',
        id='data_frame'
    ),
    dcc.Graph(id='graph_spectrum'),
    html.Br(),
    html.H2("Wybierz dane:", style={'marginTop':'20px'}),
    html.H6("Ze zbioru usunięto outliery: 8(reddish), 36, 59f.8, 98, 135f.1, 219, 250"),
    html.Div([
        dcc.Dropdown(
          options=pierwiastki.columns,
          value='Al',
          id='pierwiastek1',
          style={'width':'100px', 'marginRight':'2px'}
        ),
        dcc.Dropdown(
          options=pierwiastki.columns,
          value='Al',
          id='pierwiastek2',
          style={'width':'100px'}
        )], style={'display': 'flex'}),
    dcc.RadioItems(
        options=[
            {'label': 'dane nieznormalizowane', 'value': 'pierwiastki'},
            {'label': 'dane znormalizowane do rodu', 'value': 'pierwiastki_rh'},
            {'label': 'dane znormalizowane do Comptona', 'value': 'pierwiastki_compton'}
          ],
        value='pierwiastki',
        id='data_frame2'
        ),
    dcc.Graph(id='graph_pairplot')],
    fluid=True,
    className="px-3"
    )

In [7]:
@app.callback(
    Output('graph_spectrum', 'figure'),
    Input('sample','value'),
    Input('data_frame', 'value')
)
def draw_single(sample_list, data_frame_name):
  # Map the string ID to the actual DataFrame
  data_frames = {'dane': dane, 'dane_rh': dane_rh, 'dane_compton': dane_compton}
  selected_df = data_frames[data_frame_name]
  elements={'Al':1.49, 'Si':1.73, 'S':2.31, 'K':3.31, 'Ca':3.69, 'Ti':4.50, 'Mn':5.89, 'Fe':6.39, 'Cu':8.02, 'Zn':8.64, 'Hg':9.98, 'Pb':10.55, 'Sr':14.12}
  fig=go.Figure()
  for i in sample_list:
    fig.add_trace((go.Scatter(x=energia, y=selected_df[i], name=i)))
    fig.update_xaxes(title_text='E [KeV]', range=(0,16))
    fig.update_yaxes(title_text='counts')
    fig.update_layout(width=1200, height=600)
    if len(sample_list)==1:
      fig.update_layout(title=i)

  for key, value in elements.items():
    fig.add_vline(x=value,line_width=6, opacity=0.2)
    fig.add_annotation(text=key,
                  yref="paper",
                  x=value, y=1.05, showarrow=False)


  return fig

@app.callback(
    Output('graph_pairplot', 'figure'),
    Input('pierwiastek1','value'),
    Input('pierwiastek2', 'value'),
    Input('data_frame2', 'value')
)

def draw_pair(pierwiastek1, pierwiastek2, dane):
  data_frames2 = {'pierwiastki': pierwiastki, 'pierwiastki_rh':  pierwiastki_rh, 'pierwiastki_compton':  pierwiastki_compton}
  selected_df2 = data_frames2[dane]
  fig=px.scatter(data_frame=selected_df2, x=pierwiastek1, y=pierwiastek2, hover_name=selected_df2.index)
  fig.update_layout(width=600, height=500)
  fig.update_xaxes(title_text=pierwiastek1)
  fig.update_yaxes(title_text=pierwiastek2)

  return fig



In [8]:

app.run(jupyter_mode="external", debug=True)


OSError: Address 'http://127.0.0.1:8050' already in use.
    Try passing a different port to run.